# [SITCOM-1593] - M1M3 Force actuator analysis for different elevations

Following [SITCOM-1593], we want to plot the forces actuator errors as a function of elevation. 

Given a time range, this notebook will plot the average primaryCylinderFollowingError as well as the maximum and minimum, for different elevations. 

[SITCOM-1593]: https://rubinobs.atlassian.net/browse/SITCOM-1593

In [ ]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from astropy.time import Time
from pathlib import Path

from lsst.summit.utils.efdUtils import EfdClient, getEfdData
from lsst.sitcom.vandv import m1m3
from lsst.ts.xml.tables.m1m3 import FATable
from lsst.ts.xml.enums.MTM1M3 import DetailedStates

In [ ]:
client = EfdClient("usdf_efd")

In [ ]:
start = Time("2024-11-09 02:15:0Z", scale="utc")
end = Time("2024-11-09 05:15:0Z", scale="utc")
topic = f"lsst.sal.MTM1M3.forceActuatorData"

In [ ]:
plot_name = "Force Actuator Error vs Elevation"
plot_path = Path("./plots")
plot_path.mkdir(exist_ok=True, parents=True)

### retrieve data from EFD

In [ ]:
# Tested with up to 3h of data in USDF
# Retrieve Force Actuator information
n_actuators = 156
primary_FA_error = [
    str("".join(("primaryCylinderFollowingError", str(i)))) for i in range(n_actuators)
]
df_primary_FA_error = await client.select_time_series(
    "lsst.sal.MTM1M3.forceActuatorData", primary_FA_error, start, end
)  
# Retrieve elevations
elevations = await client.select_time_series(
    "lsst.sal.MTMount.elevation", "actualPosition", start, end
)


In [ ]:
# Retrieve detailed state from system
# this is to check that M1M3 is in an active state
# left here as reminder but commented out, you might want to check in Chronograph
# that M1M3 previous state is ACTIVE or ACTIVEENGINEERING

#def get_previous_logged_detailedState(df_state, timestamp):#
#    """
#    Get logged detailedState from M1M3 immediately before arbitrary time
#    Args:
#       df_state (pandas dataframe): pandas dataframe obtained from  time series of
#          "lsst.sal.MTM1M3.logevent_detailedState" covering a wide time frame which includes
#          the time stamp
#       timestamp (pandas timestamp): a timestamp where we want to probe the current status of M1M3
#    Returns:
#       prev_state: human readable status of current M1M3 status
#    """
#    df_state_names = df_state["detailedState"].map(lambda x: DetailedStates(x).name)
#    previous_index = df_state.index.asof(timestamp)
#    try:
#        prev = df_state.index.get_loc(previous_index)
#    except KeyError:
#        return "KeyError"
#    return df_state_names[prev]

#df_state = await client.select_time_series(
#    "lsst.sal.MTM1M3.logevent_detailedState",
#    "*",
#    t0, ##has to be previous to start
#    t1, ##has to be later than end
#)

# previous_state = get_previous_logged_detailedState(df_state, start)

# then I would need another function to check that state does not change in the (start, end) period
# or implement the following condition:
#condition = (
#    df_state["detailedStateName"] == "ACTIVE"
#)  # or (df_state["detailedStateName"] == "ACTIVEENGINEERING")
#df_state_active = df_state[condition]
#when_active = df_state_active.index.tz_localize("UTC").tz_convert(
#    df_primary_FA_error_resampled.index.tz
#)
#df_primary_FA_error_resampled_active = df_primary_FA_error_resampled.loc[
#    df_primary_FA_error_resampled.index >= when_active[0]
#]
#elevations_resampled_active = elevations_resampled.loc[
#    elevations_resampled.index >= when_active[0]
#]

### resample the data into more manageable time chunks

In [ ]:
resample_in_sec = 60  # seconds
# take the mean value for each of the actuators in {resample_in_sec} second samples
df_primary_FA_error_resampled_mean = df_primary_FA_error.resample(
    f"{resample_in_sec}s"
).mean()
# take the maximum value for each of the actuators in those {resample_in_sec} second samples
df_primary_FA_error_resampled_max = df_primary_FA_error.resample(
    f"{resample_in_sec}s"
).max()
# take the minimum value for each of the actuators in those {resample_in_sec} second samples
df_primary_FA_error_resampled_min = df_primary_FA_error.resample(
    f"{resample_in_sec}s"
).min()
# mean value of elevation in the time period ({resample_in_sec} seconds)
elevations_resampled = (
    elevations["actualPosition"].resample(f"{resample_in_sec}s").mean()
)

### obtain average, maximum and minimum across the 156 actuators

In [ ]:
average_across_actuators_resampled = df_primary_FA_error_resampled_mean.mean(axis=1)
max_across_actuators_resampled = df_primary_FA_error_resampled_max.max(axis=1)
min_across_actuators_resampled = df_primary_FA_error_resampled_min.min(axis=1)
# get the actuator ID which has a maximum or minimum value in the resampled data set
# with maximum and minimum values of each actuator
max_actuators = np.argmax(df_primary_FA_error_resampled_max, axis=1)
min_actuators = np.argmin(df_primary_FA_error_resampled_min, axis=1)

### make a scatter plot of force actuator errors for different elevations
Note that some elevation values might be repeated or slightly offset from other, similar values as the sequence of elevation stops can change a lot

In [ ]:
# choose the number of bins in which we want to group the elevation_resampled values (x axis)
n_bins = 30
bin_edges = np.linspace(
    np.min(elevations_resampled), np.max(elevations_resampled), n_bins + 1
)
bin_indices = np.digitize(elevations_resampled, bin_edges)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
plt.scatter(
    elevations_resampled,
    average_across_actuators_resampled,
    alpha=0.3,
    color="blue",
    label=f"Average value per {resample_in_sec}s period",
    s=5,
)
plt.scatter(
    elevations_resampled,
    max_across_actuators_resampled,
    alpha=0.3,
    color="orange",
    label=f"Largest value per {resample_in_sec}s period",
    s=5,
)
plt.scatter(
    elevations_resampled,
    min_across_actuators_resampled,
    alpha=0.3,
    color="red",
    label=f"Lowest value per {resample_in_sec}s period",
    s=5,
)
plt.title(plot_name, y=1.08)
t0 = pd.to_datetime(start.datetime, utc=True)
t1 = pd.to_datetime(end.datetime, utc=True)
plt.suptitle(f"{t0} - {t1}", y=0.93, fontsize=10, color="gray")
plt.ylabel("primaryCylinderFollowingError (N)")
plt.xlabel("elevation (degrees)")
plt.legend()
plt.savefig(plot_path / "sitcom-1593_fa_error_vs_elevation_scatter.png")

### histogram the actuator ID that hits the maximum and minimum record most often

In [ ]:
minhist = plt.hist(
    min_actuators, bins=156, range=[0, 156], color="red", label="Minimum"
)
maxhist = plt.hist(
    max_actuators, bins=156, range=[0, 156], color="orange", label="Maximum"
)
plt.xlabel("Actuator ID")
plt.legend()
plt.savefig(plot_path / "sitcom-1593_actuator_id_hist.png")

### plot all actuator force error behavior as a function of elevation

In [ ]:
n_bins = 30
binned_primary_FA_error_resampled_mean = [None] * n_actuators
binned_primary_FA_error_resampled_max = [None] * n_actuators
binned_primary_FA_error_resampled_min = [None] * n_actuators
bin_edges = np.linspace(
    np.min(elevations_resampled), np.max(elevations_resampled), n_bins + 1
)
bin_indices = np.digitize(elevations_resampled, bin_edges)
for j in range(n_actuators):
    binned_primary_FA_error_resampled_mean[j] = [
        df_primary_FA_error_resampled_mean[bin_indices == i][
            f"primaryCylinderFollowingError{j}"
        ].mean()
        for i in range(1, len(bin_edges))
    ]
    binned_primary_FA_error_resampled_max[j] = [
        df_primary_FA_error_resampled_max[bin_indices == i][
            f"primaryCylinderFollowingError{j}"
        ].mean()
        for i in range(1, len(bin_edges))
    ]
    binned_primary_FA_error_resampled_min[j] = [
        df_primary_FA_error_resampled_min[bin_indices == i][
            f"primaryCylinderFollowingError{j}"
        ].mean()
        for i in range(1, len(bin_edges))
    ]
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2


In [ ]:
for j in range(n_actuators):
    plt.plot(
        bin_centers,
        binned_primary_FA_error_resampled_mean[j],
        alpha=0.1,
        color="gray",
    )

    plt.plot(
        bin_centers,
        binned_primary_FA_error_resampled_min[j],
        alpha=0.1,
        color="red",
    )

    plt.plot(
        bin_centers,
        binned_primary_FA_error_resampled_max[j],
        alpha=0.1,
        color="orange",
    )

plt.title(f"{plot_name}: all actuators", y=1.08)
t0 = pd.to_datetime(start.datetime, utc=True)
t1 = pd.to_datetime(end.datetime, utc=True)
plt.suptitle(f"{t0} - {t1}", y=0.93, fontsize=10, color="gray")
plt.ylabel("primaryCylinderFollowingError (N)")
plt.xlabel("elevation (degrees)")
plt.legend()
plt.savefig(plot_path / "sitcom-1593_fa_error_vs_elevation_line.png")

### add interactivity to the plot
So that we can identify with hover the actuator IDs

In [ ]:
from collections import defaultdict
from bokeh.plotting import figure, show
from bokeh.models import HoverTool, TapTool
from bokeh.layouts import column
from bokeh.models import ColumnDataSource
from bokeh.io import output_notebook

fadata_mean = defaultdict(list)
fadata_max = defaultdict(list)
fadata_min = defaultdict(list)

fa_mean = np.array(binned_primary_FA_error_resampled_mean)
fa_max = np.array(binned_primary_FA_error_resampled_max)
fa_min = np.array(binned_primary_FA_error_resampled_min)

for i in range(n_actuators):
    fadata_mean["elevation"].append(bin_centers)
    fadata_mean["force error"].append(fa_mean[i])
    fadata_mean["FA"].append(f"Force Actuator {i}")
    fadata_max["elevation"].append(bin_centers)
    fadata_max["force error"].append(fa_max[i])
    fadata_max["FA"].append(f"Force Actuator {i}")
    fadata_min["elevation"].append(bin_centers)
    fadata_min["force error"].append(fa_min[i])
    fadata_min["FA"].append(f"Force Actuator {i}")

hover_opts = dict(tooltips=[("FA", "@FA")], show_arrow=False, line_policy="next")
line_opts_mean = dict(
    line_width=1,
    line_color="grey",
    line_alpha=0.1,
    hover_line_alpha=1.0,
    source=fadata_mean,
)
line_opts_max = dict(
    line_width=1,
    line_color="orange",
    line_alpha=0.1,
    hover_line_alpha=1.0,
    source=fadata_max,
)
line_opts_min = dict(
    line_width=1,
    line_color="red",
    line_alpha=0.1,
    hover_line_alpha=1.0,
    source=fadata_min,
)

p = figure(
    title=f"{plot_name}: all actuators",
    x_axis_label="elevation (degrees)",
    y_axis_label="primaryCylinderFollowingError (N)",
    tools=[HoverTool(**hover_opts), TapTool()],
)

p.multi_line(xs="elevation", ys="force error", **line_opts_mean)
p.multi_line(xs="elevation", ys="force error", **line_opts_max)
p.multi_line(xs="elevation", ys="force error", **line_opts_min)

output_notebook()
show(p)